# The following is the new code:

NOTE: need a CUDA GPU for bnb. CPU is not compatible with this.

Versions used:

    bitsandbytes version:  0.48.1
    torch version:  2.8.0+cu126

Installs:

In [8]:
!pip install bitsandbytes
!pip install torch
!pip install tqdm

Imports:

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import bitsandbytes as bnb
from torch.utils.data import Dataset, DataLoader
import json
from tqdm import tqdm

In [2]:
print(f'bitsandbytes version: ', bnb.__version__)
print(f'torch version: ', torch.__version__)

bitsandbytes version:  0.48.1
torch version:  2.9.0+cu128


Dataset:

Defines our dataset for future use.

In [3]:
class ChessDataset(Dataset):
    def __init__(self, tensor_data):
        self.data = tensor_data

    def __len__(self):
        return self.data.size(0)

    def __getitem__(self, idx):
        x = self.data[idx, :-1]
        y = self.data[idx, 1:]
        return x, y

Sparse Multihead Attention:

This is a sparse self-attention layer that limits how far each token can attend in the past using a sliding window (sparsity window).
This makes computation more efficient :D

In [4]:
class SparseMultiheadAttention(nn.Module):
    def __init__(self, d_model, nhead, sparsity_window=32):
        super().__init__()
        self.nhead = nhead
        self.sparsity_window = sparsity_window
        self.attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)

    def forward(self, x):
        T = x.size(1)

        # Sparse mask: prevent attending to tokens beyond a sliding window

        mask = torch.ones((T, T), device=x.device, dtype=torch.bool)
        mask = torch.triu(torch.ones(T,T,device=x.device, dtype=torch.bool), diagonal=1)

        if self.sparsity_window < T:
            for i in range(T):
                start = max(0, i - self.sparsity_window)
                mask[i, :start] = True

        out, _ = self.attn(x, x, x, attn_mask=mask)
        out = torch.nan_to_num(out)
        return out

Sparse Decode Layer:

A layer of a transformer decoder block involving sparse attn.

In [6]:
class SparseDecoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=2048, sparsity_window=32, dropout=0.1):
        super().__init__()
        self.self_attn = SparseMultiheadAttention(d_model, nhead, sparsity_window)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        attn_out = self.self_attn(x)
        x = x + self.dropout(attn_out)
        x = self.norm1(x)

        ff_out = self.linear2(F.relu(self.linear1(x)))
        x = x + self.dropout(ff_out)
        x = self.norm2(x)

        return x

Decoder:

Overall the decoder architecture (just with sparse attn).

In [7]:
class ExpandedAttentionChessDecoder(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, num_layers, max_len, sparsity_window=32):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

        self.layers = nn.ModuleList([
            SparseDecoderLayer(d_model, nhead, sparsity_window=sparsity_window)
            for _ in range(num_layers)
        ])

        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        B, T = x.size()
        positions = torch.arange(0, T, device=x.device).unsqueeze(0)
        x = self.embed(x) + self.pos_emb(positions)

        for layer in self.layers:
            x = layer(x)

        return self.fc_out(x)

Quantized Model:

This just replaces the nn.Linear layers with 8-bit versions from bnb.
Makes things more efficient :D

In [8]:
def quantize_model_8bit(model):
    for name, module in model.named_children():

        if isinstance(module, nn.Linear):
            setattr(model, name, bnb.nn.Linear8bitLt(
                module.in_features,
                module.out_features,
                bias=module.bias is not None
            ))

        else:
            quantize_model_8bit(module)

    return model

# Hyperparameter Optimization
We will use optuna's functionality on our datadset to tune hyperparameters before training on our larger dataset.

In [15]:
from torch.utils.data import random_split, DataLoader

with open("move_to_id.json", "r") as f:
    move_to_id = json.load(f)
vocab_size = len(move_to_id)
PAD_ID = move_to_id['<PAD>']
encoded_tensor = torch.load("encoded_games_small.pt")
dataset = ChessDataset(encoded_tensor)


train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)

import optuna
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device {device} found')
vocab_size = len(move_to_id)

def objective(trial):
    # search space
    d_model = trial.suggest_categorical('d_model', [128, 256, 512])
    valid_heads = [h for h in [4, 8] if d_model % h == 0]
    nhead = trial.suggest_categorical('nhead', valid_heads)
    num_layers = trial.suggest_int('num_layers', 2, 6)
    lr = trial.suggest_float('lr', 1e-5, 1e-3, log=True)

    # create model
    model = ExpandedAttentionChessDecoder(
        vocab_size=vocab_size,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        max_len=encoded_tensor.size(1),
        sparsity_window=32
    ).to(device)
    model.train()

    optimizer = optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

    # train a few epochs
    EPOCHS = 5
    for epoch in range(EPOCHS):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits.reshape(-1, vocab_size), y.reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            

    # validation
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for x, y, in val_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits.reshape(-1,vocab_size), y.reshape(-1))
            total_val_loss += loss.item()
    avg_val_loss = total_val_loss / len(val_loader)

    return avg_val_loss


device cuda found


run optimization with optuna

In [16]:
# set up optuna

from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

# run a few rounds at random first
sampler = TPESampler(n_startup_trials=5)
pruner = MedianPruner(n_startup_trials=5, n_warmup_steps=1)

study = optuna.create_study(direction='minimize', sampler=sampler, pruner=pruner)
study.optimize(objective, n_trials=20)

# print results

print('\n\nBest trial:')
print('  Value (val loss):', study.best_trial.value)
print('  Params:', study.best_trial.params)

# save best hyperparameters to file

import json
from datetime import datetime

best_params = study.best_trial.params
best_value = study.best_trial.value

best_params_with_meta = {
    "best_params": best_params,
    "best_val_loss": best_value,
    "n_trials": len(study.trials),
}

filename = f'best_hparams_ea.json'

with open(filename, 'w') as f:
    json.dump(best_params_with_meta, f)

[I 2025-10-25 10:02:44,736] A new study created in memory with name: no-name-8ded8910-80e1-4ea1-b69c-3bf4f1f3dd7f
[I 2025-10-25 10:34:39,117] Trial 0 finished with value: 3.3758561458343115 and parameters: {'d_model': 512, 'nhead': 8, 'num_layers': 4, 'lr': 0.0002052669682871821}. Best is trial 0 with value: 3.3758561458343115.
[I 2025-10-25 11:01:21,227] Trial 1 finished with value: 3.7683119651598806 and parameters: {'d_model': 256, 'nhead': 8, 'num_layers': 4, 'lr': 0.0001703113665196842}. Best is trial 0 with value: 3.3758561458343115.
[I 2025-10-25 11:31:50,791] Trial 2 finished with value: 3.5756139021653395 and parameters: {'d_model': 256, 'nhead': 4, 'num_layers': 5, 'lr': 0.0002456059519002655}. Best is trial 0 with value: 3.3758561458343115.
[I 2025-10-25 11:55:18,827] Trial 3 finished with value: 4.187309668614314 and parameters: {'d_model': 512, 'nhead': 4, 'num_layers': 3, 'lr': 4.1724090605538245e-05}. Best is trial 0 with value: 3.3758561458343115.
[I 2025-10-25 12:20:54



Best trial:
  Value (val loss): 3.0286924013724694
  Params: {'d_model': 512, 'nhead': 8, 'num_layers': 6, 'lr': 0.0007126937796402232}


# Model Training

Note: Set batch size, sequence length, num layers, d_model, epochs to initial settings.


Also, remove "encoded_tensor = encoded_tensor[:, :256]  # FOR TESTING --- REMOVE LATER!!!". This was for testing. It speeds it up a lot...

You may also need to reset the dataset being used depending on what you want to use.


BTW this is where the training is (shocker).
This is what trains our newly quantized, sparse attention decoder transformer on the training dataset.

In [17]:
if __name__ == "__main__":
    with open("move_to_id.json", "r") as f:
        move_to_id = json.load(f)
    vocab_size = len(move_to_id)
    PAD_ID = move_to_id['<PAD>']
    encoded_tensor = torch.load("encoded_games_500k.pt")
    dataset = ChessDataset(encoded_tensor)
    loader = DataLoader(dataset, batch_size=64, shuffle=True, drop_last=True)


    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Running on device: {device}")

    with open("best_hparams_ea.json", "r") as f:
        best_hparams = json.load(f)

    params = best_hparams["best_params"]

    # Build model on CPU first
    model = ExpandedAttentionChessDecoder(
        vocab_size=vocab_size,
        d_model=params["d_model"],  # Commented out for testing
        nhead=params["nhead"],
        num_layers=params["num_layers"],  # Commented out for testing
        max_len=encoded_tensor.size(1),
        sparsity_window=32  # can adjust to 32 for even higher sparsity
    )

    # Quantize while still on CPU
    model = quantize_model_8bit(model)

    # Now move to GPU
    model = model.to(device)
    model.train()
    print("Model quantized to 8-bit and moved to CUDA with sparse attention.")

    optimizer = bnb.optim.Adam8bit(model.parameters(), lr=params["lr"])
    print("Using bnb Adam8bit optimizer.")

    # Training!
    EPOCHS = 10

    for epoch in range(EPOCHS):
      epoch_loss = 0
      num_batches = 0

      progress_bar = tqdm(loader, desc=f"Epoch {epoch+1}", leave=True)
      for x, y in progress_bar:
          x, y = x.to(device), y.to(device)

          logits = model(x)
          loss = F.cross_entropy(
              logits.reshape(-1, vocab_size),
              y.reshape(-1),
              ignore_index=PAD_ID
          )

          optimizer.zero_grad()
          loss.backward()
          optimizer.step()

          epoch_loss += loss.item()
          num_batches += 1
          avg_loss = epoch_loss / num_batches

          progress_bar.set_postfix(loss=f"{avg_loss:.4f}")

      print(f"Epoch {epoch+1}: avg loss = {avg_loss:.4f}")

      torch.save(model.state_dict(), f"quant_8bit_epoch{epoch+1}.pt")

      print("Saved quantized model checkpoint.")

Running on device: cuda
Model quantized to 8-bit and moved to CUDA with sparse attention.
Using bnb Adam8bit optimizer.


Epoch 1:   0%|          | 0/15595 [00:00<?, ?it/s]/home/dsu/Desktop/Mason_Wyatt/venv/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
Epoch 1: 100%|██████████| 15595/15595 [3:08:36<00:00,  1.38it/s, loss=4.0192]  


Epoch 1: avg loss = 4.0192
Saved quantized model checkpoint.


Epoch 2: 100%|██████████| 15595/15595 [3:04:02<00:00,  1.41it/s, loss=3.3718]  


Epoch 2: avg loss = 3.3718
Saved quantized model checkpoint.


...

Epoch 4: avg loss = 3.1952
Saved quantized model checkpoint.


Epoch 5: 100%|██████████| 15595/15595 [3:20:06<00:00,  1.30it/s, loss=3.1601]  


Epoch 5: avg loss = 3.1601
Saved quantized model checkpoint.


Epoch 6:  84%|████████▎ | 13039/15595 [2:50:21<33:23,  1.28it/s, loss=3.1345]  


KeyboardInterrupt: 